In [1]:
import pandas as pd
import re
import io

print("NYC LAYOVER CHATBOT - DATASET PREPARATION")

NYC LAYOVER CHATBOT - DATASET PREPARATION


In [2]:
# STEP 1: LOAD RAW COMMONPLACE DATA
# Load the full raw dataset
df_raw = pd.read_csv('rawdata.csv')

In [3]:
# STEP 2: MANUALLY CURATED ATTRACTION DATA
manual_data_csv = """name,category,entry_cost,visit_duration_hrs
METROPOLITAN MUSEUM OF ART,Museums & Galleries,$30,3.0
TIMES SQUARE MUSEUM & VISITOR CENTER,Museums & Galleries,FREE,0.5
STATUE OF LIBERTY MUSEUM,Museums & Galleries,FREE,1.5
9/11 MEMORIAL MUSEUM,Museums & Galleries,$33,2.0
AMERICAN MUSEUM OF NATURAL HISTORY,Museums & Galleries,$28,3.0
THE THEATER AT MADISON SQUARE GARDEN,Shows & Nightlife,$50-300,3.0
MUSEUM OF MODERN ART (MOMA),Museums & Galleries,$25,2.5
WHITNEY MUSEUM OF AMERICAN ART,Museums & Galleries,$30,2.0
MET CLOISTERS MUSEUM METROPOLITAN MUSEUM OF ART,Museums & Galleries,$30,3.0
SOLOMON R GUGGENHEIM MUSEUM,Museums & Galleries,$25,2.0
INTREPID SEA AIR & SPACE MUSEUM,Museums & Galleries,$36,2.5
MADISON SQUARE GARDEN,Shows & Nightlife,$50-300,3.0
ROCKEFELLER CENTER ICE RINK,Historical & Cultural Sites; Landmarks & Viewpoints,$25-35,1.5
GRAND CENTRAL LIBRARY,Historical & Cultural Sites; Landmarks & Viewpoints,FREE,0.5
BRONX ZOO ROCKEFELLER FOUNTAIN,Historical & Cultural Sites; Landmarks & Viewpoints,FREE,0.25
LINCOLN CENTER METROPOLITAN OPERA HOUSE,Shows & Nightlife,$40-250,3.0
RADIO CITY MUSIC HALL,Shows & Nightlife,$50-150,2.5
APOLLO THEATRE 125 ST,Shows & Nightlife,$30-75,2.5
YANKEE STADIUM,Shows & Nightlife,$20-200,3.5
EMPIRE STATE BUILDING,Historical & Cultural Sites; Landmarks & Viewpoints,$44,2.0
YANKEE TEAM CLUBHOUSE YANKEE STADIUM,Shows & Nightlife,$20-200,3.5
CARNEGIE HALL,Shows & Nightlife,$30-150,2.5
BROOKLYN MUSEUM OF ART,Museums & Galleries,$20,2.0
CHELSEA MARKET,Food & Drink,FREE,1.5
TKTS TIMES SQUARE,Historical & Cultural Sites; Landmarks & Viewpoints,FREE,0.5
HIGH LINE W 34 ST ENTRANCE,Nature & Adventure,FREE,1.5
LEGENDS SUITE CLUB YANKEE STADIUM,Shows & Nightlife,$20-200,3.5
BROOKLYN BRIDGE PARK MAIN STREET PARK,Historical & Cultural Sites; Landmarks & Viewpoints,FREE,1.0
MOHEGAN SUN SPORTS BAR YANKEE STADIUM,Shows & Nightlife,$20-200,3.5
HARD ROCK CAFE YANKEE STADIUM,Food & Drink,$20-40,1.5
YANKEE CLUBHOUSE STORE YANKEE STADIUM,Shows & Nightlife,$20-200,3.5
NYY STEAKHOUSE YANKEE STADIUM,Shows & Nightlife,$20-200,3.5
BRYANT PARK,Nature & Adventure,FREE,1.0
GRAND CENTRAL TERMINAL,Historical & Cultural Sites; Landmarks & Viewpoints,FREE,0.5
STATUE OF LIBERTY,Historical & Cultural Sites; Landmarks & Viewpoints,$24,3.5
STATUE OF LIBERTY & ELLIS ISLAND TERMINAL,Historical & Cultural Sites; Landmarks & Viewpoints,$24,3.5
THE BIG BAT YANKEE STADIUM,Shows & Nightlife,$20-200,3.5
MUSEUM OF SEX,Museums & Galleries,$30,1.5
MUSEUM OF THE DOG,Museums & Galleries,$15-20,1.5
NATIONAL MUSEUM OF MATHEMATICS,Museums & Galleries,$20,1.5
TRANSIT MUSEUM GALLERY ANNEX & STORE,Museums & Galleries,FREE,0.5
TRIENNALE DESIGN MUSEUM,Museums & Galleries,$15-20,1.5
MUSEUM OF ILLUSIONS,Museums & Galleries,$29,1.0
MUSEUM OF AMERICAN FOLK ART,Museums & Galleries,$15-20,1.5
SHELBY CULLOM DAVIS MUSEUM,Museums & Galleries,$15-20,1.5
MADAME TUSSAUD'S WAX MUSEUM,Museums & Galleries,$45,1.5
MUSEUM OF ARTS AND DESIGN,Museums & Galleries,$18,1.5
MUSEUM OF CONTEMPORARY ART,Museums & Galleries,$15-20,1.5
MORGAN LIBRARY & MUSEUM,Museums & Galleries,$22,1.5
ASIA SOCIETY & MUSEUM,Museums & Galleries,$15-20,1.5
PALACE THEATRE,Shows & Nightlife,$50-200,2.5
FORNINO RESTAURANT BROOKLYN BRIDGE PARK,Food & Drink,$20-40,1.5
CHRYSLER BUILDING,Historical & Cultural Sites; Landmarks & Viewpoints,FREE,0.5
CENTRAL PARK ZOO GALLERY,Museums & Galleries,$15-20,1.5
JEFFERSON MARKET LIBRARY,Shopping,FREE,0.5
JEFFERSON MARKET GARDEN,Shopping,FREE,0.5
PROSPECT PARK ZOO SEA LION STORE & CAFE,Food & Drink,$20-40,1.5
BRONX ZOO-DANCING CRANE CAFE,Food & Drink,$20-40,1.5
WHOLE FOODS MARKET UNION SQUARE,Food & Drink,FREE,1.5
BRONX ZOO - BRONX ZOO STORE,Shopping,FREE,1.0
PARK AVENUE MALLS,Shopping,FREE,1.5
BROADWAY MALLS,Shopping,FREE,1.5
WASHINGTON MARKET PARK,Shopping,FREE,1.0
CENTRAL PARK MALL,Shopping,FREE,1.5
DANCING CRANE CAFE,Food & Drink,$20-40,1.5
BROOKLYN BOTANICAL GARDEN CAFE,Food & Drink,$20-40,1.5
ESSEX MARKET,Food & Drink,FREE,1.5
VERNON MALL,Shopping,FREE,1.5
SCHIFF MALLS,Shopping,FREE,1.5
FDR DRIVE MALLS,Shopping,FREE,1.5
HARD ROCK CAFE,Food & Drink,$20-40,1.5
EATALY FLATIRON,Food & Drink,$20-40,2.0
ROCK CENTER CAFE,Food & Drink,$20-40,1.5
PERSHING SQUARE CAFE,Food & Drink,$20-40,1.5
NIOS RESTAURANT,Food & Drink,$20-40,1.5
NIPPON RESTAURANT,Food & Drink,$20-40,1.5
VAUCLUSE RESTAURANT,Food & Drink,$20-40,1.5
PERFECT PINT RESTAURANT,Food & Drink,$20-40,1.5
JACK DEMSEYS RESTAURANT,Food & Drink,$20-40,1.5
HUDSON YARDS SHOPS AND RESTAURANTS,Food & Drink,$20-40,1.5
CHELSEA SQUARE RESTAURANT,Food & Drink,$20-40,1.5"""

# Load manual data
df_manual = pd.read_csv(io.StringIO(manual_data_csv))

# Fix category column (semicolon instead of comma for some entries)
df_manual['category'] = df_manual['category'].str.replace(';', ',')

In [4]:
# STEP 3: FILTER RAW DATA & MERGE WITH MANUAL DATA

# Extract coordinates from geometry field
def extract_coords(geom_str):
    """Extract latitude and longitude from POINT geometry string"""
    if pd.isna(geom_str):
        return None, None
    match = re.search(r'POINT \(([-.0-9]+) ([-.0-9]+)\)', str(geom_str))
    if match:
        return float(match.group(2)), float(match.group(1))  # lat, lon
    return None, None

df_raw[['latitude', 'longitude']] = df_raw['the_geom'].apply(
    lambda x: pd.Series(extract_coords(x))
)

# Normalize feature names for matching
df_raw['name_normalized'] = df_raw['FEATURE NAME'].str.upper().str.strip()
df_manual['name_normalized'] = df_manual['name'].str.upper().str.strip()

# Create set of attraction names
attraction_names = set(df_manual['name_normalized'])

# Filter raw data
df_filtered = df_raw[df_raw['name_normalized'].isin(attraction_names)].copy()

# Merge with manual data
df = df_filtered.merge(
    df_manual,
    on='name_normalized',
    how='inner'
)

# Select columns
df = df[[
    'name', 'category', 'latitude', 'longitude',
    'entry_cost', 'visit_duration_hrs'
]]

if len(df) != 81:
    print(f"\n⚠️  WARNING: Expected 81 attractions, got {len(df)}")
    print("   Some attractions may not have matched.")

In [5]:
# Calculate distances

JFK_LAT, JFK_LON = 40.6413, -73.7781
LGA_LAT, LGA_LON = 40.7769, -73.8740
EWR_LAT, EWR_LON = 40.6895, -74.1745
MANHATTAN_LAT, MANHATTAN_LON = 40.7589, -73.9851

def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two GPS coordinates"""
    R = 6371  # Earth radius in km
    import math
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

df['distance_jfk_km'] = df.apply(
    lambda row: haversine(JFK_LAT, JFK_LON, row['latitude'], row['longitude']),
    axis=1
)
df['distance_lga_km'] = df.apply(
    lambda row: haversine(LGA_LAT, LGA_LON, row['latitude'], row['longitude']),
    axis=1
)
df['distance_ewr_km'] = df.apply(
    lambda row: haversine(EWR_LAT, EWR_LON, row['latitude'], row['longitude']),
    axis=1
)
df['distance_from_manhattan_km'] = df.apply(
    lambda row: haversine(MANHATTAN_LAT, MANHATTAN_LON, row['latitude'], row['longitude']),
    axis=1
)

# Add popularity score (based on category and distance from Manhattan)
df['popularity_score'] = 100 + (5 - df['distance_from_manhattan_km']) * 10

In [6]:
# STEP 4: ADD BUDGET LEVEL COLUMN (FROM COST)

def assign_budget_level(entry_cost):
    """
    Convert entry_cost to budget_level (Low/Medium/High)
    
    Budget tiers:
    - Low: FREE - $30
    - Medium: $31 - $100
    - High: $100+
    """
    
    if pd.isna(entry_cost) or entry_cost == 'FREE':
        return 'Low'
    
    # Extract maximum cost from string
    cost_str = str(entry_cost).replace('$', '').strip()
    
    # Handle ranges like "$20-40"
    if '-' in cost_str:
        parts = cost_str.split('-')
        try:
            max_cost = float(parts[1])
        except:
            max_cost = 0
    else:
        try:
            max_cost = float(cost_str)
        except:
            max_cost = 0
    
    # Classify into tiers
    if max_cost == 0:
        return 'Low'
    elif max_cost <= 30:
        return 'Low'
    elif max_cost <= 100:
        return 'Medium'
    else:
        return 'High'

# Apply function
df['budget_level'] = df['entry_cost'].apply(assign_budget_level)

print("\n  Distribution:")
for level in ['Low', 'Medium', 'High']:
    count = len(df[df['budget_level'] == level])
    pct = (count / len(df)) * 100 if len(df) > 0 else 0
    print(f"    {level:8} {count:3} attractions ({pct:5.1f}%)")


  Distribution:
    Low       45 attractions ( 55.6%)
    Medium    23 attractions ( 28.4%)
    High      13 attractions ( 16.0%)


In [7]:
# STEP 5: ADD NEIGHBORHOOD COLUMN (FROM COORDINATES)

def assign_neighborhood(row):
    """Assign neighborhood based on GPS coordinates"""
    
    lat = row['latitude']
    lon = row['longitude']
    name = str(row['name']).upper()
    
    # MIDTOWN MANHATTAN
    if 40.745 <= lat <= 40.77 and -74.01 <= lon <= -73.96:
        return 'Midtown'
    
    # UPPER EAST SIDE
    if lat > 40.76 and lon > -73.97:
        return 'Upper East Side'
    
    # UPPER WEST SIDE
    if lat > 40.76 and lon < -73.97:
        return 'Upper West Side'
    
    # CHELSEA
    if 40.74 <= lat <= 40.755 and lon < -73.99:
        return 'Chelsea'
    
    # GREENWICH VILLAGE
    if 40.72 <= lat <= 40.74 and -74.01 <= lon <= -73.99:
        return 'Greenwich Village'
    
    # DOWNTOWN
    if lat < 40.72 and lon < -73.99:
        return 'Downtown'
    
    # BROOKLYN
    if 'BROOKLYN' in name or (lat < 40.73 and lon > -73.99):
        return 'Brooklyn'
    
    # BRONX
    if 'BRONX' in name or 'YANKEE' in name or lat > 40.82:
        return 'Bronx'
    
    # Default
    return 'Manhattan Other'

# Apply function
df['neighborhood'] = df.apply(assign_neighborhood, axis=1)

print("\n  Distribution:")
for hood in sorted(df['neighborhood'].unique()):
    count = len(df[df['neighborhood'] == hood])
    print(f"    {hood:25} {count:3} attractions")


  Distribution:
    Brooklyn                    4 attractions
    Chelsea                     1 attractions
    Downtown                    8 attractions
    Greenwich Village           5 attractions
    Manhattan Other             4 attractions
    Midtown                    37 attractions
    Upper East Side            16 attractions
    Upper West Side             6 attractions


In [8]:
# STEP 6: ADD TIME SLOT COLUMNS (FROM CATEGORY)

def assign_time_slots(row):
    """Assign time slot availability based on category"""
    
    category = row['category']
    name = str(row['name']).upper()
    
    # OUTDOOR 24/7 LANDMARKS - Check FIRST before category
    outdoor_24_7_keywords = [
        'TIMES SQUARE', 'BROOKLYN BRIDGE', 'HIGH LINE', 'BRYANT PARK',
        'CENTRAL PARK', 'WASHINGTON SQUARE', 'UNION SQUARE',
        'BATTERY PARK', 'CHARGING BULL', 'WALL STREET'
    ]
    
    # Exclude indoor facilities that happen to have these names
    indoor_exclusions = ['MUSEUM', 'VISITOR CENTER', 'GALLERY', 'ZOO', 
                         'STORE', 'CAFE', 'RESTAURANT', 'TERMINAL']
    
    # Check if it's an outdoor 24/7 landmark
    if any(keyword in name for keyword in outdoor_24_7_keywords):
        if not any(indoor in name for indoor in indoor_exclusions):
            return pd.Series({'slot_daytime': 1, 'slot_evening': 1, 'slot_allday': 1})
    
    # MUSEUMS & GALLERIES - daytime only
    if category == 'Museums & Galleries':
        return pd.Series({'slot_daytime': 1, 'slot_evening': 0, 'slot_allday': 0})
    
    # SHOWS & NIGHTLIFE - evening only
    if category == 'Shows & Nightlife':
        return pd.Series({'slot_daytime': 0, 'slot_evening': 1, 'slot_allday': 0})
    
    # HISTORICAL SITES - check if outdoor
    if 'Historical & Cultural Sites' in category:
        # Outdoor landmarks = all day (but NOT buildings/terminals)
        if any(k in name for k in ['BRIDGE', 'SQUARE', 'PARK', 'PLAZA', 'FOUNTAIN']):
            if not any(indoor in name for indoor in ['TERMINAL', 'BUILDING', 
                                                      'TOWER', 'CENTER', 'LIBRARY']):
                return pd.Series({'slot_daytime': 1, 'slot_evening': 1, 'slot_allday': 1})
        # Buildings = daytime only
        return pd.Series({'slot_daytime': 1, 'slot_evening': 0, 'slot_allday': 0})
    
    # FOOD & DRINK
    if category == 'Food & Drink':
        # Markets close by evening
        if 'MARKET' in name:
            return pd.Series({'slot_daytime': 1, 'slot_evening': 0, 'slot_allday': 0})
        # Restaurants = daytime + evening (NOT all day)
        return pd.Series({'slot_daytime': 1, 'slot_evening': 1, 'slot_allday': 0})
    
    # SHOPPING - daytime only
    if category == 'Shopping':
        return pd.Series({'slot_daytime': 1, 'slot_evening': 0, 'slot_allday': 0})
    
    # NATURE - daytime only
    if category == 'Nature & Adventure':
        return pd.Series({'slot_daytime': 1, 'slot_evening': 0, 'slot_allday': 0})
    
    # Default
    return pd.Series({'slot_daytime': 1, 'slot_evening': 0, 'slot_allday': 0})

# Apply function
df[['slot_daytime', 'slot_evening', 'slot_allday']] = df.apply(assign_time_slots, axis=1)

print(f"\n  Daytime: {len(df[df['slot_daytime'] == 1])} attractions")
print(f"  Evening: {len(df[df['slot_evening'] == 1])} attractions")
print(f"  All day: {len(df[df['slot_allday'] == 1])} attractions")


  Daytime: 67 attractions
  Evening: 38 attractions
  All day: 7 attractions


In [9]:
# STEP 7: EXPORT FINAL DATASET

# Reorder columns
column_order = [
    'name', 'category', 'latitude', 'longitude', 'neighborhood',
    'distance_jfk_km', 'distance_lga_km', 'distance_ewr_km',
    'distance_from_manhattan_km', 'popularity_score',
    'entry_cost', 'budget_level', 'visit_duration_hrs',
    'slot_daytime', 'slot_evening', 'slot_allday'
]

df_final = df[column_order].copy()

# Save
output_file = 'nyc_attractions_final.csv'
df_final.to_csv(output_file, index=False)

print(f"\n Final dataset saved: {output_file}")
print(f"  Rows: {len(df_final)}")
print(f"  Columns: {len(df_final.columns)}")


 Final dataset saved: nyc_attractions_final.csv
  Rows: 81
  Columns: 16


In [10]:
# STEP 8: SUMMARY
print(f"  Raw CommonPlace: {len(df_raw):,} facilities")
print(f"  Final dataset:   {len(df_final)} attractions")

print(f"\n Ready for chatbot: {output_file}")
print("\nColumns (16 total):")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:2}. {col}")
print()

  Raw CommonPlace: 20,529 facilities
  Final dataset:   81 attractions

 Ready for chatbot: nyc_attractions_final.csv

Columns (16 total):
   1. name
   2. category
   3. latitude
   4. longitude
   5. neighborhood
   6. distance_jfk_km
   7. distance_lga_km
   8. distance_ewr_km
   9. distance_from_manhattan_km
  10. popularity_score
  11. entry_cost
  12. budget_level
  13. visit_duration_hrs
  14. slot_daytime
  15. slot_evening
  16. slot_allday

